# MarketMaven - Colab Training & Evaluation

This notebook replicates the full MarketMaven training pipeline on Google Colab,
giving you GPU-accelerated training with interactive verification before exporting
artifacts back to your local codebase.

## Workflow
1. **Setup** - Install deps, clone repo, verify GPU
2. **Config** - Load `phase_0.yaml`, override device to `cuda`
3. **Data** - Fetch OHLCV from yfinance, build features, split, normalize
4. **Train** - Multi-seed training (5 seeds) with full logging
5. **Evaluate** - Test-set metrics + backtest for each seed
6. **Visualize** - Loss curves, equity curves, seed comparison
7. **Export** - Download verified artifacts as a zip

## Prerequisites
- Set runtime to **GPU** (Runtime > Change runtime type > T4 GPU)
- If repo is private, set `GITHUB_TOKEN` in the config cell below

---
## 1. Environment Setup

In [1]:
#@title 1a. Check GPU Availability
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU available: {gpu_name} ({gpu_mem:.1f} GB)')
    print(f'CUDA version: {torch.version.cuda}')
else:
    print('WARNING: No GPU detected. Training will be slow.')
    print('Go to Runtime > Change runtime type > select T4 GPU')

print(f'PyTorch version: {torch.__version__}')

Go to Runtime > Change runtime type > select T4 GPU
PyTorch version: 2.10.0


In [3]:
#@title 1b. Install Dependencies (training-only subset)
import subprocess
import sys

import torch

packages = ['yfinance', 'pyarrow', 'pyyaml', 'joblib', 'scikit-learn', 'einops']

# Auto-install CUDA Mamba kernels when a CUDA runtime is available.
# This prevents slow pure-PyTorch fallback for Phase 2 (mamba_ssm).
if torch.cuda.is_available():
    packages.append('mamba-ssm')

print('Installing dependencies...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', *packages],
    capture_output=True,
    text=True,
    check=False,
)

if result.returncode != 0:
    print('Dependency installation failed:')
    print(result.stderr)
    raise RuntimeError('pip install failed; see stderr above.')

print('Dependencies installed.')

if torch.cuda.is_available():
    try:
        from mamba_ssm import Mamba  # noqa: F401

        print('mamba-ssm available: CUDA backend should be enabled for Phase 2.')
    except Exception as exc:
        print('WARNING: mamba-ssm import failed after install; Phase 2 will use slow fallback.')
        print(f'Reason: {exc}')


Installing dependencies...
Dependencies installed.


In [5]:
#@title 1c. Clone Repository { display-mode: "form" }
import os
import shutil
import subprocess
from pathlib import Path


def _running_in_colab() -> bool:
    return bool(os.environ.get('COLAB_RELEASE_TAG'))


def _default_repo_dir() -> str:
    # Colab uses /content (writable). Local Mac/Linux cannot use /content (read-only or missing).
    if _running_in_colab():
        return '/content/MarketMaven-BE'
    return str((Path.cwd() / 'MarketMaven-BE').resolve())


def _find_existing_repo(start: Path) -> Path | None:
    for p in (start, *start.parents):
        if (p / 'pyproject.toml').is_file() and (p / 'api').is_dir():
            return p
    return None


#@markdown **Repository URL** (HTTPS)
REPO_URL = 'https://github.com/jayantdahiya/MarketMaven-BE.git' #@param {type:"string"}

#@markdown **GitHub Personal Access Token** (leave empty for public repos)
GITHUB_TOKEN = '' #@param {type:"string"}

#@markdown **Branch to checkout** (leave empty for default branch)
BRANCH = 'develop' #@param {type:"string"}

#@markdown **Clone destination** (leave empty: Colab → `/content/MarketMaven-BE`, local → `./MarketMaven-BE`)
REPO_DIR_OVERRIDE = '' #@param {type:"string"}

#@markdown **Local only:** if the notebook lives inside a MarketMaven-BE checkout, reuse it (no clone)
USE_LOCAL_REPO_IF_FOUND = True #@param {type:"boolean"}

REPO_DIR = REPO_DIR_OVERRIDE.strip() or _default_repo_dir()
repo_path = Path(REPO_DIR).resolve()
reused_local = False

if USE_LOCAL_REPO_IF_FOUND and not _running_in_colab():
    found = _find_existing_repo(Path.cwd())
    if found is not None:
        REPO_DIR = str(found.resolve())
        repo_path = found.resolve()
        reused_local = True
        print(f'Using existing repo at {REPO_DIR} (skipping clone).')
        if BRANCH:
            subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
        result = subprocess.run(
            ['git', 'log', '--oneline', '-3'], cwd=REPO_DIR,
            capture_output=True, text=True, check=False,
        )
        print(result.stdout)

os.environ['MARKETMAVEN_REPO_DIR'] = str(repo_path)

if not reused_local:
    # Ensure parent directory exists and is writable (local Jupyter)
    parent = repo_path.parent
    parent.mkdir(parents=True, exist_ok=True)
    if not os.access(parent, os.W_OK):
        raise RuntimeError(
            f'Cannot write to {parent}. Set REPO_DIR_OVERRIDE to a writable path.'
        )

    if repo_path.exists():
        shutil.rmtree(repo_path)

    if GITHUB_TOKEN:
        clone_url = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@')
    else:
        clone_url = REPO_URL

    clone_result = subprocess.run(
        ['git', 'clone', clone_url, str(repo_path)],
        capture_output=True,
        text=True,
        check=False,
    )
    if clone_result.returncode != 0:
        print(clone_result.stderr)
        raise RuntimeError('git clone failed; see stderr above.')

    if BRANCH:
        subprocess.run(['git', 'checkout', BRANCH], cwd=str(repo_path), check=True)

    print(f'\nCloned to {repo_path}')

    result = subprocess.run(
        ['git', 'log', '--oneline', '-3'], cwd=str(repo_path),
        capture_output=True, text=True, check=False,
    )
    print(result.stdout)

Using existing repo at /Users/Work/Desktop/Personal/MarketMaven-BE (skipping clone).
M	notebooks/colab_training.ipynb
Your branch is up to date with 'origin/develop'.
218121c docs: update README and colab_training notebook for improved setup instructions
e9dac57 fix: flush stale api namespace from sys.modules for Colab compatibility
eade05c docs: enhance AGENTS.md with proactive library usage guidelines and update pyproject.toml for development dependencies



Already on 'develop'


In [ ]:
#@title 1d. Install repo as editable package & verify imports
import importlib
import os
import subprocess
import sys
from pathlib import Path

REPO_DIR = os.environ.get('MARKETMAVEN_REPO_DIR', '/content/MarketMaven-BE')
if not Path(REPO_DIR).is_dir():
    raise RuntimeError(
        f'REPO_DIR {REPO_DIR!r} is missing. Run the clone cell (1c) first, or set '
        'MARKETMAVEN_REPO_DIR to your checkout root.'
    )
os.chdir(REPO_DIR)

# Ensure package markers exist for api + subpackages. This prevents
# namespace-package collisions in Colab (e.g., with preinstalled 'api').
pkg_dirs = [
    Path(REPO_DIR) / 'api',
    Path(REPO_DIR) / 'api' / 'data',
    Path(REPO_DIR) / 'api' / 'models',
    Path(REPO_DIR) / 'api' / 'training',
]
for pkg_dir in pkg_dirs:
    if not pkg_dir.exists():
        print(f'Skipping missing package dir: {pkg_dir}')
        continue
    init_path = pkg_dir / '__init__.py'
    if not init_path.exists():
        init_path.touch()
        print(f'Created {init_path}')

# Ensure repo root takes priority during imports.
if REPO_DIR in sys.path:
    sys.path.remove(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# Install the repo itself as an editable package (--no-deps to avoid
# pulling heavy unused deps like prophet, supabase, redis, etc.).
# This registers 'api' as a proper package so Colab's pre-installed
# namespace packages (google.api, etc.) cannot shadow our submodules.
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps', '-q'],
    capture_output=True, text=True, check=False,
)
if result.returncode != 0:
    print('pip install -e . failed:')
    print(result.stderr)
    raise RuntimeError('Editable install failed; see stderr above.')
print('Editable install complete.')

# Flush any cached 'api' namespace from sys.modules. Colab pre-loads
# google-api-core and other packages that register 'api' as a namespace
# package. Even after pip install -e ., the stale in-memory cache
# prevents Python from finding our api.models / api.training, etc.
stale = [k for k in sys.modules if k == 'api' or k.startswith('api.')]
for k in stale:
    del sys.modules[k]
importlib.invalidate_caches()
if stale:
    print(f'Cleared {len(stale)} stale api.* module(s) from cache.')

# Verify imports work
from api.data.pipeline import DataPipeline, load_config
from api.models.factory import create_model
from api.training.train_loop import Trainer
from api.training import losses, evaluate

import api
print(f"api imported from: {getattr(api, '__file__', '<namespace>')}")
print('All imports successful.')

---
## 2. Configuration

In [ ]:
#@title 2a. Load & Override Config { display-mode: "form" }
from pathlib import Path

#@markdown ### Training Overrides
#@markdown Adjust these as needed. The config file defaults are used for anything not overridden.

CONFIG_PATH = 'config/phase_0.yaml' #@param {type:"string"}
DEVICE = 'cuda' #@param ["cuda", "cpu"] {type:"string"}
EPOCHS = 50 #@param {type:"integer"}
BATCH_SIZE = 64 #@param {type:"integer"}
LEARNING_RATE = 0.001 #@param {type:"number"}
EARLY_STOPPING_PATIENCE = 10 #@param {type:"integer"}

#@markdown ### Mamba (Phase 2) Runtime Controls
MAMBA_BACKEND = 'auto' #@param ["auto", "cuda", "pure_pytorch"] {type:"string"}
MAMBA_STABILITY_MODE = True #@param {type:"boolean"}

#@markdown ### Path Handling
#@markdown When enabled, checkpoint/report paths follow config project phase automatically.
AUTO_PHASE_PATHS = True #@param {type:"boolean"}
STRICT_PHASE_PATHS = True #@param {type:"boolean"}

#@markdown Optional explicit path overrides (leave empty to auto/config defaults)
DATA_DIR_OVERRIDE = '' #@param {type:"string"}
CHECKPOINTS_DIR_OVERRIDE = '' #@param {type:"string"}
REPORTS_DIR_OVERRIDE = '' #@param {type:"string"}

# Load config with base inheritance
cfg = load_config(CONFIG_PATH)
cfg.setdefault('training', {})
cfg.setdefault('paths', {})

# Override device and training params for Colab
cfg['training']['device'] = DEVICE
cfg['training']['epochs'] = EPOCHS
cfg['training']['batch_size'] = BATCH_SIZE
cfg['training']['lr'] = LEARNING_RATE
cfg['training']['early_stopping_patience'] = EARLY_STOPPING_PATIENCE

# Model-specific runtime overrides
cfg.setdefault('mamba', {})
if cfg.get('model', {}).get('type') == 'mamba_ssm':
    cfg['mamba']['backend'] = MAMBA_BACKEND

    # Stability guard for Colab Phase 2 training.
    # Helps prevent NaN explosions seen with aggressive one-cycle + Sharpe.
    if MAMBA_STABILITY_MODE:
        cfg['training']['lr'] = min(cfg['training'].get('lr', LEARNING_RATE), 3e-4)
        cfg['training']['grad_clip_norm'] = min(cfg['training'].get('grad_clip_norm', 1.0), 0.5)
        cfg['training']['early_stopping_patience'] = min(cfg['training'].get('early_stopping_patience', EARLY_STOPPING_PATIENCE), 8)

        loss_cfg = cfg['training'].setdefault('loss', {})
        loss_cfg['lambda_sharpe'] = min(loss_cfg.get('lambda_sharpe', 0.0), 0.05)
        loss_cfg['warmup_epochs'] = max(loss_cfg.get('warmup_epochs', 0), 5)

        sched_cfg = cfg['training'].setdefault('scheduler', {})
        if sched_cfg.get('type') == 'one_cycle_lr':
            sched_cfg['max_lr'] = min(sched_cfg.get('max_lr', cfg['training']['lr']), cfg['training']['lr'])


# Resolve phase-aware paths
phase_num = cfg.get('project', {}).get('phase', 0)
phase_tag = f'phase{phase_num}'
default_data_dir = cfg['paths'].get('data_dir', 'artifacts/data/phase0')
default_checkpoints_dir = cfg['paths'].get('checkpoints_dir', f'artifacts/checkpoints/{phase_tag}')
default_reports_dir = cfg['paths'].get('reports_dir', f'artifacts/reports/{phase_tag}')

resolved_data_dir = DATA_DIR_OVERRIDE.strip() or default_data_dir
if AUTO_PHASE_PATHS:
    resolved_checkpoints_dir = CHECKPOINTS_DIR_OVERRIDE.strip() or f'artifacts/checkpoints/{phase_tag}'
    resolved_reports_dir = REPORTS_DIR_OVERRIDE.strip() or f'artifacts/reports/{phase_tag}'
else:
    resolved_checkpoints_dir = CHECKPOINTS_DIR_OVERRIDE.strip() or default_checkpoints_dir
    resolved_reports_dir = REPORTS_DIR_OVERRIDE.strip() or default_reports_dir

# Strict guardrail: avoid accidental phase0 writes for non-phase0 runs
if STRICT_PHASE_PATHS and phase_num != 0:
    if resolved_checkpoints_dir.endswith('/phase0') or '/phase0/' in resolved_checkpoints_dir:
        raise ValueError(
            f'checkpoints_dir resolved to {resolved_checkpoints_dir!r} for phase {phase_num}. '
            'Set CHECKPOINTS_DIR_OVERRIDE or keep AUTO_PHASE_PATHS enabled.'
        )
    if resolved_reports_dir.endswith('/phase0') or '/phase0/' in resolved_reports_dir:
        raise ValueError(
            f'reports_dir resolved to {resolved_reports_dir!r} for phase {phase_num}. '
            'Set REPORTS_DIR_OVERRIDE or keep AUTO_PHASE_PATHS enabled.'
        )

cfg['paths']['data_dir'] = resolved_data_dir
cfg['paths']['checkpoints_dir'] = resolved_checkpoints_dir
cfg['paths']['reports_dir'] = resolved_reports_dir

# Ensure artifact directories exist
for d in cfg['paths'].values():
    if isinstance(d, str):
        Path(d).mkdir(parents=True, exist_ok=True)

print('=== Resolved Configuration ===')
print(f'Config:     {CONFIG_PATH}')
print(f'Phase:      {phase_num} ({phase_tag})')
print(f'Device:     {cfg["training"]["device"]}')
print(f'Model:      {cfg["model"]["type"]}')
print(f'Epochs:     {cfg["training"]["epochs"]}')
print(f'Batch size: {cfg["training"]["batch_size"]}')
print(f'LR:         {cfg["training"]["lr"]}')
print(f'Grad clip:  {cfg["training"].get("grad_clip_norm", 1.0)}')
print(f'Seeds:      {cfg["training"]["seeds"]}')
print(f'Assets:     {cfg["data"]["assets"]}')
print(f'Seq len:    {cfg["data"]["seq_len"]}')
print(f'Features:   {cfg["data"]["feature_cols"]}')
print(f'Data dir:   {cfg["paths"]["data_dir"]}')
print(f'CKPT dir:   {cfg["paths"]["checkpoints_dir"]}')
print(f'Reports:    {cfg["paths"]["reports_dir"]}')
print(f"Sharpe λ:   {cfg.get('training', {}).get('loss', {}).get('lambda_sharpe', 'n/a')}")
if cfg.get('model', {}).get('type') == 'mamba_ssm':
    print(f"Mamba backend: {cfg.get('mamba', {}).get('backend', 'auto')}")
print(f'Splits:     train<={cfg["data"]["split"]["train_end"]}, '
      f'val<={cfg["data"]["split"]["val_end"]}, '
      f'test<={cfg["data"]["split"]["test_end"]}')

---
## 3. Data Pipeline

In [ ]:
#@title 3a. Fetch Data & Build DataLoaders
import logging

import torch

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

pipeline = DataPipeline(cfg)
loaders = pipeline.build_dataloaders(shuffle_train=True)

print(f'\n=== Dataset Sizes ===')
for split_name, loader in loaders.items():
    n_samples = len(loader.dataset)
    n_batches = len(loader)
    print(f'{split_name:>5s}: {n_samples:,} samples ({n_batches} batches)')

# Sanity check: inspect one batch
x_sample, y_sample, meta_sample = next(iter(loaders['train']))
print(f'\n=== Sample Batch ===')
print(f'x shape: {x_sample.shape}  (batch, seq_len, features)')
print(f'y shape: {y_sample.shape}  (batch, horizon)')
print(f'x dtype: {x_sample.dtype}')
print(f'y range: [{y_sample.min():.4f}, {y_sample.max():.4f}]')
print(f'Any NaN in x: {torch.isnan(x_sample).any()}')
print(f'Any NaN in y: {torch.isnan(y_sample).any()}')

---
## 4. Training (Multi-Seed)

In [ ]:
#@title 4a. Training Loop - All Seeds
import random
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
from torch.optim.lr_scheduler import CosineAnnealingLR, OneCycleLR, ReduceLROnPlateau


def set_global_seed(seed: int) -> None:
    """Set seed for reproducibility across all libraries."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_scheduler(optimizer, sched_cfg, steps_per_epoch=0, epochs=1):
    """Instantiate LR scheduler from config dict."""
    sched_type = sched_cfg.get('type', 'reduce_lr_on_plateau')
    if sched_type == 'cosine_annealing':
        return CosineAnnealingLR(
            optimizer,
            T_max=sched_cfg.get('T_max', 50),
            eta_min=sched_cfg.get('eta_min', 1e-6),
        )
    if sched_type == 'reduce_lr_on_plateau':
        return ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=sched_cfg.get('factor', 0.5),
            patience=sched_cfg.get('patience', 3),
            min_lr=sched_cfg.get('min_lr', 1e-6),
        )
    if sched_type == 'one_cycle_lr':
        if steps_per_epoch <= 0:
            print(f'WARNING: OneCycleLR needs steps_per_epoch > 0; got {steps_per_epoch}')
            return None
        return OneCycleLR(
            optimizer,
            max_lr=sched_cfg.get('max_lr', 6e-4),
            total_steps=epochs * steps_per_epoch,
            pct_start=sched_cfg.get('pct_start', 0.3),
            anneal_strategy=sched_cfg.get('anneal_strategy', 'cos'),
            div_factor=sched_cfg.get('div_factor', 25.0),
            final_div_factor=sched_cfg.get('final_div_factor', 1000.0),
        )
    print(f'WARNING: Unknown scheduler type {sched_type!r}')
    return None


# ---- Prepare shared objects ----
train_cfg = cfg.get('training', {})
device = train_cfg.get('device', 'cuda')
seeds = train_cfg.get('seeds', [42, 123, 456, 789, 1024])
loss_cfg = train_cfg.get('loss', {})
sched_cfg = train_cfg.get('scheduler', {})
model_cfg = cfg.get('model', {})
model_type = model_cfg.get('type', 'lstm_baseline')
feature_cols = cfg.get('data', {}).get('feature_cols', [])
paths_cfg = cfg.get('paths', {})
scaler_path = str(Path(paths_cfg.get('data_dir', 'artifacts/data/phase0')) / 'scaler.joblib')

# Save scaler from the pipeline
if pipeline._scaler is not None:
    pipeline._scaler.save(scaler_path)
scaler_state = pipeline._scaler.get_state() if pipeline._scaler is not None else {}

# Store results for all seeds
all_results = {}  # seed -> {'result': ..., 'run_id': ...}

print(f'Starting training: {len(seeds)} seeds, {train_cfg.get("epochs", 50)} max epochs each')
print(f'Device: {device}')
print(f'Model: {model_type}')
print('=' * 70)

for i, seed in enumerate(seeds):
    set_global_seed(seed)
    run_id = f'seed_{seed}_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
    print(f'\n{"=" * 70}')
    print(f'[{i+1}/{len(seeds)}] Seed {seed} | Run ID: {run_id}')
    print(f'{"=" * 70}')

    # Build fresh model, optimizer, scheduler, criterion for each seed
    criterion = losses.CompositeForecastLoss(
        lambda_sharpe=loss_cfg.get('lambda_sharpe', 0.1),
        temperature=loss_cfg.get('temperature', 0.02),
        warmup_epochs=loss_cfg.get('warmup_epochs', 3),
        sharpe_window=loss_cfg.get('sharpe_window', 0),
    )

    model = create_model(model_type, model_cfg)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=train_cfg.get('lr', 0.001),
        weight_decay=train_cfg.get('weight_decay', 0.0001),
    )

    scheduler = build_scheduler(
        optimizer,
        sched_cfg,
        steps_per_epoch=len(loaders['train']),
        epochs=train_cfg.get('epochs', 50),
    )

    trainer = Trainer(cfg, model, optimizer, scheduler, criterion, device)

    result = trainer.fit(
        loaders['train'],
        loaders['val'],
        run_id=run_id,
        feature_cols=feature_cols,
        scaler_path=scaler_path,
        scaler_state=scaler_state,
    )

    all_results[seed] = {'result': result, 'run_id': run_id}

    # Print summary for this seed
    history = result['history']
    best_epoch_idx = 0
    best_mae = float('inf')
    for idx, entry in enumerate(history):
        if entry['val_mae'] < best_mae:
            best_mae = entry['val_mae']
            best_epoch_idx = idx
    best_entry = history[best_epoch_idx]
    print(f'\nSeed {seed} complete:')
    print(f'  Epochs trained: {len(history)}')
    print(f'  Best epoch:     {best_entry["epoch"]}')
    print(f'  Best val MAE:   {best_entry["val_mae"]:.6f}')
    print(f'  Best val RMSE:  {best_entry["val_rmse"]:.6f}')
    print(f'  Checkpoint:     {result["best_checkpoint_path"]}')

print(f'\n{"=" * 70}')
print(f'All {len(seeds)} seeds complete!')
print(f'{"=" * 70}')

---
## 5. Evaluation on Test Set

In [ ]:
#@title 5a. Evaluate All Seeds on Test Set
import pandas as pd
from pathlib import Path

# Rebuild loaders without shuffling for consistent evaluation
eval_pipeline = DataPipeline(cfg)
eval_loaders = eval_pipeline.build_dataloaders(shuffle_train=False)
eval_scaler = eval_pipeline._scaler

# Store evaluation results
eval_results = {}  # seed -> evaluate_checkpoint output
metrics_rows = []  # for summary DataFrame

paths_cfg = cfg.get('paths', {})

for seed, info in all_results.items():
    checkpoint_path = info['result']['best_checkpoint_path']
    run_id = info['run_id']

    print(f'\nEvaluating seed {seed} | {checkpoint_path}')

    # Create fresh model for evaluation
    model = create_model(model_type, model_cfg)

    results = evaluate.evaluate_checkpoint(
        cfg, checkpoint_path, eval_loaders['test'], eval_scaler, model
    )

    eval_results[seed] = results

    # Write reports to disk
    report_dir = Path(paths_cfg.get('reports_dir', f"artifacts/reports/phase{cfg.get('project', {}).get('phase', 0)}")) / run_id
    evaluate.write_reports(
        str(report_dir),
        results['summary'],
        results['regime_df'],
        results['bt_df'],
        results['bt_summary'],
        model_tag=model_type,
    )

    # Collect metrics for summary table
    row = {'seed': seed, **results['summary']}
    metrics_rows.append(row)

    print(f'  MAE: {results["summary"]["mae"]:.5f}')
    print(f'  RMSE: {results["summary"]["rmse"]:.5f}')
    print(f'  Directional Accuracy: {results["summary"]["directional_accuracy"]:.4f}')
    print(f'  Sharpe: {results["summary"]["sharpe"]:.4f}')
    print(f'  Net PnL: {results["summary"]["net_pnl"]:.4f}')
    print(f'  Reports: {report_dir}')

# Build summary DataFrame
metrics_df = pd.DataFrame(metrics_rows)
display_cols = ['seed', 'mae', 'rmse', 'directional_accuracy', 'sharpe',
                'sortino', 'max_drawdown', 'net_pnl']
display_df = metrics_df[[c for c in display_cols if c in metrics_df.columns]]

print(f'\n{"=" * 70}')
print('EVALUATION SUMMARY (all seeds)')
print(f'{"=" * 70}')
print(display_df.to_string(index=False, float_format='%.5f'))

# Highlight the best seed
if len(display_df) > 0:
    best_idx = display_df['mae'].idxmin()
    best_seed = display_df.loc[best_idx, 'seed']
    print(f'\nBest seed by MAE: {int(best_seed)}')
    best_idx_sharpe = display_df['sharpe'].idxmax()
    best_seed_sharpe = display_df.loc[best_idx_sharpe, 'seed']
    print(f'Best seed by Sharpe: {int(best_seed_sharpe)}')

---
## 6. Visualizations

In [ ]:
#@title 6a. Training Loss Curves (per seed)
import matplotlib.pyplot as plt
from pathlib import Path

PLOT_COLORS = ['#2196F3', '#FF5722', '#4CAF50', '#9C27B0', '#FF9800']
reports_dir = Path(cfg['paths'].get('reports_dir', f"artifacts/reports/phase{cfg.get('project', {}).get('phase', 0)}"))
reports_dir.mkdir(parents=True, exist_ok=True)

n_seeds = len(all_results)
fig, axes = plt.subplots(2, n_seeds, figsize=(5 * n_seeds, 8), squeeze=False)

for col, (seed, info) in enumerate(all_results.items()):
    history = info['result']['history']
    epochs = [h['epoch'] for h in history]
    train_loss = [h['train_loss'] for h in history]
    val_loss = [h['val_loss'] for h in history]
    val_mae = [h['val_mae'] for h in history]
    val_rmse = [h['val_rmse'] for h in history]

    # Loss curves
    ax = axes[0][col]
    ax.plot(epochs, train_loss, label='Train Loss', color='#2196F3', linewidth=1.5)
    ax.plot(epochs, val_loss, label='Val Loss', color='#FF5722', linewidth=1.5)
    ax.set_title(f'Seed {seed} - Loss', fontsize=11)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    # MAE / RMSE curves
    ax2 = axes[1][col]
    ax2.plot(epochs, val_mae, label='Val MAE', color='#4CAF50', linewidth=1.5)
    ax2.plot(epochs, val_rmse, label='Val RMSE', color='#9C27B0', linewidth=1.5)
    ax2.set_title(f'Seed {seed} - Metrics', fontsize=11)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Error')
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(reports_dir / 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {reports_dir / "training_curves.png"}')

In [ ]:
#@title 6b. Equity Curves (per seed backtest)
fig, ax = plt.subplots(figsize=(14, 6))

for i, (seed, results) in enumerate(eval_results.items()):
    bt_df = results['bt_df']
    equity = bt_df['cumulative_equity'].values
    color = PLOT_COLORS[i % len(PLOT_COLORS)]
    ax.plot(range(len(equity)), equity, label=f'Seed {seed}', color=color, linewidth=1.2, alpha=0.85)

ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='Baseline (1.0)')
ax.set_title('Equity Curves - All Seeds (Test Set)', fontsize=13)
ax.set_xlabel('Trading Days')
ax.set_ylabel('Cumulative Equity')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(reports_dir / 'equity_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {reports_dir / "equity_curves.png"}')

In [ ]:
#@title 6c. Seed Comparison Bar Charts
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

bar_metrics = [
    ('mae', 'MAE (lower is better)', True),
    ('rmse', 'RMSE (lower is better)', True),
    ('directional_accuracy', 'Directional Accuracy (higher is better)', False),
    ('sharpe', 'Sharpe Ratio (higher is better)', False),
    ('sortino', 'Sortino Ratio (higher is better)', False),
    ('net_pnl', 'Net PnL (higher is better)', False),
]

seed_labels = list(eval_results.keys())  # int labels, not strings
x_positions = range(len(seed_labels))

for idx, (metric_key, title, lower_is_better) in enumerate(bar_metrics):
    ax = axes[idx // 3][idx % 3]
    values = [eval_results[s]['summary'].get(metric_key, 0) for s in seed_labels]

    # Color the best bar differently
    best_idx = values.index(min(values)) if lower_is_better else values.index(max(values))
    bar_colors = ['#90CAF9'] * len(values)
    bar_colors[best_idx] = '#2196F3'

    bars = ax.bar(x_positions, values, color=bar_colors, edgecolor='white', linewidth=0.5)
    ax.set_xticks(list(x_positions))
    ax.set_xticklabels([str(s) for s in seed_labels])
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Seed')
    ax.grid(True, alpha=0.3, axis='y')

    # Add value labels on bars
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                f'{val:.4f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Seed Comparison - Test Set Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(reports_dir / 'seed_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {reports_dir / "seed_comparison.png"}')

In [ ]:
#@title 6d. Training History - All Seeds Overlaid
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metric_configs = [
    ('val_loss', 'Validation Loss'),
    ('val_mae', 'Validation MAE'),
    ('val_rmse', 'Validation RMSE'),
]

for ax, (metric_key, title) in zip(axes, metric_configs):
    for i, (seed, info) in enumerate(all_results.items()):
        history = info['result']['history']
        epochs = [h['epoch'] for h in history]
        values = [h[metric_key] for h in history]
        color = PLOT_COLORS[i % len(PLOT_COLORS)]
        ax.plot(epochs, values, label=f'Seed {seed}', color=color, linewidth=1.2, alpha=0.85)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Epoch')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Training Convergence - All Seeds', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(reports_dir / 'convergence_overlay.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {reports_dir / "convergence_overlay.png"}')

---
## 7. Export Artifacts

In [ ]:
#@title 7a. Select Seeds to Export { display-mode: "form" }

#@markdown Choose which seeds to include in the export.
#@markdown Set to `all` to export everything, or comma-separated seed numbers (e.g. `42,123`).
SEEDS_TO_EXPORT = 'all' #@param {type:"string"}

if SEEDS_TO_EXPORT.strip().lower() == 'all':
    export_seeds = list(all_results.keys())
else:
    export_seeds = [int(s.strip()) for s in SEEDS_TO_EXPORT.split(',')]

print(f'Seeds to export: {export_seeds}')

# Show metrics for selected seeds
for seed in export_seeds:
    if seed in eval_results:
        s = eval_results[seed]['summary']
        print(f'  Seed {seed}: MAE={s["mae"]:.5f}  Sharpe={s["sharpe"]:.4f}  PnL={s["net_pnl"]:.4f}')
    else:
        print(f'  Seed {seed}: NOT FOUND in eval_results (skipping)')

In [ ]:
#@title 7b. Package & Download Artifacts
import os
import zipfile
from pathlib import Path

paths_cfg = cfg.get('paths', {})
scaler_path = str(Path(paths_cfg.get('data_dir', 'artifacts/data/phase0')) / 'scaler.joblib')
reports_dir = Path(paths_cfg.get('reports_dir', f"artifacts/reports/phase{cfg.get('project', {}).get('phase', 0)}"))

# Writable path: Colab uses /content; local Jupyter uses repo dir or cwd
if os.environ.get('COLAB_RELEASE_TAG'):
    EXPORT_ZIP = '/content/marketmaven_artifacts.zip'
elif os.environ.get('MARKETMAVEN_REPO_DIR'):
    EXPORT_ZIP = str(
        Path(os.environ['MARKETMAVEN_REPO_DIR']) / 'marketmaven_artifacts.zip'
    )
else:
    EXPORT_ZIP = str(Path.cwd() / 'marketmaven_artifacts.zip')

# Clean previous export
if os.path.exists(EXPORT_ZIP):
    os.remove(EXPORT_ZIP)

files_added = 0
added_paths = set()  # track to avoid duplicates

with zipfile.ZipFile(EXPORT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    # 1. Scaler file
    if os.path.exists(scaler_path):
        arcname = os.path.relpath(scaler_path, '.')
        zf.write(scaler_path, arcname)
        added_paths.add(arcname)
        files_added += 1
        print(f'Added: {arcname}')

    for seed in export_seeds:
        if seed not in all_results:
            print(f'WARNING: Seed {seed} not found, skipping.')
            continue

        info = all_results[seed]
        run_id = info['run_id']
        best_ckpt = info['result']['best_checkpoint_path']

        # 2. Best checkpoint .pt file
        if os.path.exists(best_ckpt):
            arcname = os.path.relpath(best_ckpt, '.')
            zf.write(best_ckpt, arcname)
            added_paths.add(arcname)
            files_added += 1
            print(f'Added: {arcname}')

        # 3. last.pt checkpoint
        last_ckpt = str(Path(best_ckpt).parent / 'last.pt')
        if os.path.exists(last_ckpt):
            arcname = os.path.relpath(last_ckpt, '.')
            zf.write(last_ckpt, arcname)
            added_paths.add(arcname)
            files_added += 1
            print(f'Added: {arcname}')

        # 4. Report files
        report_dir = reports_dir / run_id
        if report_dir.exists():
            for report_file in report_dir.iterdir():
                if report_file.is_file():
                    arcname = os.path.relpath(str(report_file), '.')
                    if arcname not in added_paths:
                        zf.write(str(report_file), arcname)
                        added_paths.add(arcname)
                        files_added += 1
                        print(f'Added: {arcname}')

    # 5. Visualization PNGs (top-level, not inside seed dirs)
    for png in reports_dir.glob('*.png'):
        arcname = os.path.relpath(str(png), '.')
        if arcname not in added_paths:
            zf.write(str(png), arcname)
            added_paths.add(arcname)
            files_added += 1
            print(f'Added: {arcname}')

zip_size_mb = os.path.getsize(EXPORT_ZIP) / 1e6
print(f'\nExport complete: {files_added} files, {zip_size_mb:.1f} MB')
print(f'ZIP: {EXPORT_ZIP}')

In [ ]:
#@title 7c. Download the ZIP
try:
    from google.colab import files
    files.download(EXPORT_ZIP)
    print('Download started. Check your browser downloads.')
except ImportError:
    print(f'Not running in Colab. ZIP is at: {EXPORT_ZIP}')
    print('Download it manually or copy from the file browser.')

---
## 8. Import Instructions (Local Machine)

After downloading `marketmaven_artifacts.zip`, unzip it **from your repo root**:

```bash
cd /path/to/MarketMaven-BE
unzip marketmaven_artifacts.zip
```

This will place files in the correct locations (based on your selected phase/config):
```
artifacts/
  data/<phase_data>/scaler.joblib                         # Fitted scaler (from cfg['paths']['data_dir'])
  checkpoints/phase<k>/seed_42_YYYYMMDD_HHMMSS/          # Per-seed dirs (phase-aware)
    best_epoch_XXX.pt                                     # Best checkpoint
    last.pt                                               # Final checkpoint
  reports/phase<k>/seed_42_YYYYMMDD_HHMMSS/              # Per-seed reports (phase-aware)
    metrics_summary.json
    regime_metrics.csv
    backtest_report.csv
    backtest_summary.json
```

The API's `ForecastService` picks up checkpoints from the model's configured
checkpoint directory (for LSTM this is usually `artifacts/checkpoints/phase0`,
for CNN-Transformer `artifacts/checkpoints/phase1`, etc.).

### Verify locally
```bash
# Check the checkpoint loads correctly
python scripts/evaluate.py --config config/phase_1.yaml \
    --checkpoint artifacts/checkpoints/phase1/seed_42_YYYYMMDD_HHMMSS/best_epoch_XXX.pt

# Start the API (will use the new checkpoint)
python api/main.py
```

### Notes
- Checkpoints trained on CUDA load fine on CPU/MPS (handled by `map_location` in `load_checkpoint`)
- Phase 1 can intentionally reuse Phase 0 data dir (`artifacts/data/phase0`)
- The scaler state is embedded in the `.pt` checkpoint, so `scaler.joblib` is a backup
- Reports are for your reference; the API does not read them